In [3]:
!git clone https://github.com/Uyen-nt/MTG-downstreamtask.git

Cloning into 'MTG-downstreamtask'...
remote: Enumerating objects: 3269, done.
remote: Counting objects: 100% (230/230), done.
remote: Compressing objects: 100% (195/195), done.
remote: Total 3269 (delta 170), reused 35 (delta 35), pack-reused 3039 (from 3)
Receiving objects: 100% (3269/3269), 1.22 GiB | 48.04 MiB/s, done.
Resolving deltas: 100% (1961/1961), done.
Updating files: 100% (286/286), done.


In [4]:
%cd MTG-downstreamtask

/kaggle/working/MTG-downstreamtask


In [2]:
!git pull origin main

fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [5]:
!python gpt/data_preprocessing.py

Loading CSVs Into Dataframes
Building Dataset
100%|████████████████████████████████████| 12911/12911 [01:10<00:00, 184.02it/s]
VOCAB SIZE: 4252
Adding Labels
Converting Visits
MAX LEN: 34
AVG LEN: 1.2911
MAX VISIT LEN: 39
AVG VISIT LEN: 9.160870575478274
NUM RECORDS: 10000
NUM LONGITUDINAL RECORDS: 1688
Splitting Datasets
Saving Everything


In [ ]:
#!git pull origin main

In [ ]:
#!python -m gpt.train

In [6]:
import os
import torch
import pickle
import random
import numpy as np
from tqdm import tqdm
from sklearn import metrics
from gpt.model import GPTModel
from gpt.config import GPTConfig
import torch.nn.functional as F

SEED = 4
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
config = GPTConfig()

# ==========================
# DEVICE
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# ==========================
# LOAD DATASET & MAPPINGS
# Đường dẫn đúng trong Kaggle
# ==========================

train_ehr_dataset = pickle.load(open('/kaggle/working/MTG-downstreamtask/gpt/result/trainDataset.pkl', 'rb'))
index_to_code = pickle.load(open("/kaggle/working/MTG-downstreamtask/gpt/result/indexToCode.pkl", "rb"))

# ==========================
# ADD LABEL NAMES
# ==========================
index_to_code[config.code_vocab_size] = "Chronic Condition: Alzheimer or related disorders or senile"
index_to_code[config.code_vocab_size+1] = "Chronic Condition: Heart Failure"
index_to_code[config.code_vocab_size+2] = "Chronic Condition: Chronic Kidney Disease"
index_to_code[config.code_vocab_size+3] = "Chronic Condition: Cancer"
index_to_code[config.code_vocab_size+4] = "Chronic Condition: Chronic Obstructive Pulmonary Disease"
index_to_code[config.code_vocab_size+5] = "Chronic Condition: Depression"
index_to_code[config.code_vocab_size+6] = "Chronic Condition: Diabetes"
index_to_code[config.code_vocab_size+7] = "Chronic Condition: Ischemic Heart Disease"
index_to_code[config.code_vocab_size+8] = "Chronic Condition: Osteoporosis"
index_to_code[config.code_vocab_size+9] = "Chronic Condition: rheumatoid arthritis and osteoarthritis (RA/OA)"
index_to_code[config.code_vocab_size+10] = "Chronic Condition: Stroke/transient Ischemic Attack"

# ==========================
# LOAD MODEL
# ==========================
model = GPTModel(config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

print("🔍 Loading pretrained GPT model...")
checkpoint = torch.load('/kaggle/input/gpt-mimic/gpt_model_best (1).pt', map_location=torch.device(device), weights_only=False)
model.load_state_dict(checkpoint['model'])
print("✔ Model weights loaded successfully")

# ==========================
# SAMPLING
# ==========================
def sample_sequence(model, length, context, batch_size=None, device='cuda', sample=True):
  context = torch.tensor(context, device=device, dtype=torch.long).unsqueeze(0).repeat(batch_size, 1)
  prev = context
  ehr = context
  past = None
  with torch.no_grad():
    for _ in range(length):
      code_logits, past = model(prev, past=past)
      code_logits = code_logits[:, -1, :]
      log_probs = F.softmax(code_logits, dim=-1)
      if sample:
        prev = torch.multinomial(log_probs, num_samples=1)
      else:
        prev = torch.argmax(log_probs, dim=1)
      ehr = torch.cat((ehr, prev), dim=1)
      
      if all([config.code_vocab_size + config.label_vocab_size + 3 in ehr[i] for i in range(batch_size)]):
        break
  ehr = ehr.cpu().detach().numpy()
  next = None
  prev = None
  return ehr

# ==========================
# CONVERT MODEL OUTPUT TO TEXT
# ==========================
def convert_ehr(ehrs, index_to_code=None):
  ehr_outputs = []
  for i in range(len(ehrs)):
    ehr = ehrs[i]
    ehr_output = []
    visit_output = []
    labels_output = np.zeros(config.label_vocab_size)
    started_visits = False
    for j in range(1, len(ehr)):
      code = ehr[j]
      if not started_visits:
        if code == config.code_vocab_size + config.label_vocab_size + 1:
          started_visits = True
        elif code >= config.code_vocab_size and code < config.code_vocab_size + config.label_vocab_size:
          labels_output[code - config.code_vocab_size] = 1
          
      else:
        if code < config.code_vocab_size:
          if code not in visit_output:
            visit_output.append(index_to_code[code] if index_to_code is not None else code)
        elif code == config.code_vocab_size + config.label_vocab_size + 2:
          if visit_output != []:
            ehr_output.append(visit_output)
            visit_output = []
        elif code == config.code_vocab_size + config.label_vocab_size + 3:
          break
        
    if visit_output != []:
      ehr_output.append(visit_output)
      
    if index_to_code is not None:
      labels_output = [index_to_code[idx + config.code_vocab_size] for idx in np.nonzero(labels_output)[0]]

    ehr_outputs.append({'visits': ehr_output, 'labels': labels_output})
  ehr = None
  ehr_output = None
  labels_output = None
  visit_output = None
  return ehr_outputs


# ==========================
# RUN GENERATION
# ==========================

synthetic_ehr_dataset = []
stoken = [config.code_vocab_size+config.label_vocab_size]

print("🚀 Generating synthetic EHR...")
for i in tqdm(range(0, len(train_ehr_dataset), 2*config.batch_size)):
  bs = min([len(train_ehr_dataset)-i, 2*config.batch_size])
  batch_synthetic_ehrs = sample_sequence(model, config.n_ctx, stoken, batch_size=bs, device=device, sample=True)
  batch_synthetic_ehrs = convert_ehr(batch_synthetic_ehrs)
  synthetic_ehr_dataset += batch_synthetic_ehrs

# ==========================
# SAVE OUTPUT
# ==========================
pickle.dump(
    synthetic_ehr_dataset,
    open('/kaggle/working/MTG-downstreamtask/gpt/result/gptDataset.pkl', 'wb')
)

print("🎉 DONE — saved synthetic dataset to /kaggle/working/MTG-downstreamtask/gpt/result/gptDataset.pkl")


DEVICE: cuda
🔍 Loading pretrained GPT model...
✔ Model weights loaded successfully
🚀 Generating synthetic EHR...


100%|██████████| 225/225 [01:39<00:00,  2.25it/s]


🎉 DONE — saved synthetic dataset to /kaggle/working/MTG-downstreamtask/gpt/result/gptDataset.pkl


In [7]:
import pickle

path = "/kaggle/working/MTG-downstreamtask/gpt/result/gptDataset.pkl"
data = pickle.load(open(path, "rb"))

print(type(data))
print("Số lượng mẫu:", len(data))

print("Ví dụ phần tử đầu tiên:")
print(data[0])


<class 'list'>
Số lượng mẫu: 7200
Ví dụ phần tử đầu tiên:
{'visits': [[1893, 1861, 2407, 3569, 2552, 3322], [709, 3136, 4036, 170, 3757, 50, 541, 1331, 2552, 3322, 987, 390, 2794, 1199, 379], [2473, 3531, 3119, 4079, 1396, 1331, 2649, 3322, 1001, 124, 3169, 2407, 3539, 1560, 989, 4158, 2559]], 'labels': array([1., 0., 0., 1., 1., 0., 1., 0., 1., 1., 1., 1., 1., 1., 0., 1., 1.,
       0., 0., 0., 0., 0., 0., 1., 1.])}


In [11]:
import pickle
import numpy as np
from scipy.spatial.distance import jensenshannon as jsd

# =============================
# CONFIG
# =============================
GPT_SYN_PATH = "/kaggle/working/MTG-downstreamtask/gpt/result/gptDataset.pkl"
REAL_PATH = "/kaggle/working/MTG-downstreamtask/gpt/result/trainDataset.pkl"
CODE_TO_INDEX = "/kaggle/working/MTG-downstreamtask/gpt/result/codeToIndex.pkl"
INDEX_TO_CODE = "/kaggle/working/MTG-downstreamtask/gpt/result/indexToCode.pkl"

# =============================
# Load required data
# =============================
print("Loading data...")

gpt_data = pickle.load(open(GPT_SYN_PATH, "rb"))
real_data = pickle.load(open(REAL_PATH, "rb"))
code_to_index = pickle.load(open(CODE_TO_INDEX, "rb"))
index_to_code = pickle.load(open(INDEX_TO_CODE, "rb"))

print("Vocabulary size:", len(code_to_index))
print("GPT samples:", len(gpt_data))
print("REAL samples:", len(real_data))

# =========================================================
# 1) Convert REAL into multi-hot format
# =========================================================

def convert_real_to_multihot(real_data, code_vocab_size):
    max_visit_len = max(len(p['visits']) for p in real_data)
    N = len(real_data)

    X = np.zeros((N, max_visit_len, code_vocab_size), dtype=np.int8)
    lens = np.zeros((N,), dtype=np.int32)

    for i, rec in enumerate(real_data):
        visits = rec['visits']
        lens[i] = len(visits)

        for j, visit in enumerate(visits):
            for code in visit:
                X[i, j, code] = 1

    return X, lens

print("\nConverting REAL...")
real_x, real_lens = convert_real_to_multihot(real_data, len(code_to_index))
print("REAL format:", real_x.shape, real_lens.shape)


# =========================================================
# 2) Convert GPT synthetic into multihot — using mapping
# =========================================================

def convert_gpt_to_multihot(gpt_data, index_to_code_GPT, code_to_index_REAL, code_vocab_size):
    max_visit_len = max(len(p['visits']) for p in gpt_data)
    N = len(gpt_data)

    X = np.zeros((N, max_visit_len, code_vocab_size), dtype=np.int8)
    lens = np.zeros((N,), dtype=np.int32)

    missing_count = 0

    for i, rec in enumerate(gpt_data):
        visits = rec['visits']
        lens[i] = len(visits)

        for j, visit in enumerate(visits):
            for gpt_code_idx in visit:
                code_str = index_to_code_GPT.get(gpt_code_idx)

                if code_str not in code_to_index_REAL:
                    missing_count += 1
                    continue

                real_code_idx = code_to_index_REAL[code_str]
                X[i, j, real_code_idx] = 1

    print(f"[INFO] GPT codes missing in REAL vocab: {missing_count}")

    return X, lens


print("\nConverting GPT synthetic...")
fake_x, fake_lens = convert_gpt_to_multihot(gpt_data, index_to_code, code_to_index, len(code_to_index))
print("FAKE format:", fake_x.shape, fake_lens.shape)


# =========================================================
# 3) Distribution & evaluation functions
# =========================================================

def get_distribution(data, lens, code_num):
    p_count = {}
    v_dist = np.zeros((code_num, ))
    p_dist = np.zeros((code_num,))

    for i, (p, L) in enumerate(zip(data, lens)):
        for v in range(L):
            codes = np.where(p[v] > 0)[0]
            for c in codes:
                v_dist[c] += 1
                if c in p_count:
                    p_count[c].add(i)
                else:
                    p_count[c] = {i}

    v_dist /= v_dist.sum()

    for c, s in p_count.items():
        p_dist[c] = len(s)
    p_dist /= p_dist.sum()

    return v_dist, p_dist


def normalized_distance(dist1, dist2):
    mask = (dist1 + dist2) > 0
    d1 = dist1[mask]
    d2 = dist2[mask]
    ratio = np.abs(d1 - d2) / ((d1 + d2) / 2)
    return np.mean(ratio)


def calc_distance(real_x, real_lens, fake_x, fake_lens, code_num):
    rv, rp = get_distribution(real_x, real_lens, code_num)
    fv, fp = get_distribution(fake_x, fake_lens, code_num)
    jsd_v = jsd(rv, fv)
    nd_v = normalized_distance(rv, fv)
    jsd_p = jsd(rp, fp)
    nd_p = normalized_distance(rp, fp)
    return jsd_v, jsd_p, nd_v, nd_p


# =========================================================
# 4) RUN EVALUATION
# =========================================================
print("\nRunning JSD & ND evaluation...")

jsd_v, jsd_p, nd_v, nd_p = calc_distance(real_x, real_lens, fake_x, fake_lens, len(code_to_index))

print("\n================= EVALUATION RESULT =================")
print(f"JSD_v (visit dist):  {jsd_v:.6f}")
print(f"JSD_p (patient dist): {jsd_p:.6f}")
print(f"ND_v  (visit dist):  {nd_v:.6f}")
print(f"ND_p  (patient dist): {nd_p:.6f}")
print("=====================================================")


# =========================================================
# 5) Print TOP-20 most common codes — REAL vs FAKE
# =========================================================

def get_code_counts(x, lens):
    counts = {}
    for p, L in zip(x, lens):
        for v in range(L):
            codes = np.where(p[v] > 0)[0]
            for c in codes:
                counts[c] = counts.get(c, 0) + 1
    sorted_count = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    return sorted_count

real_count = get_code_counts(real_x, real_lens)
fake_count = get_code_counts(fake_x, fake_lens)

print("\n=========== TOP-20 CODES REAL ===========")
for c, cnt in real_count[:20]:
    print(index_to_code[c], cnt)

print("\n=========== TOP-20 CODES FAKE ===========")
for c, cnt in fake_count[:20]:
    print(index_to_code[c], cnt)


Loading data...
Vocabulary size: 4252
GPT samples: 7200
REAL samples: 7200

Converting REAL...
REAL format: (7200, 34, 4252) (7200,)

Converting GPT synthetic...
[INFO] GPT codes missing in REAL vocab: 1
FAKE format: (7200, 11, 4252) (7200,)

Running JSD & ND evaluation...

================= EVALUATION RESULT =================
JSD_v (visit dist):  0.714686
JSD_p (patient dist): 0.709648
ND_v  (visit dist):  1.413684
ND_p  (patient dist): 1.407994

=========== TOP-20 CODES REAL ===========
4019 2801
4280 2006
41401 1770
42731 1750
V053 1419
V290 1370
25000 1186
5849 1134
51881 1038
2720 1031
5990 900
V3000 870
486 714
V3001 667
2859 636
53081 627
5070 606
496 603
2449 594
7742 569

=========== TOP-20 CODES FAKE ===========
4550 3260
9980 2611
72633 2318
E9352 2067
53240 1104
4148 1089
1414 1071
V3001 1046
75610 993
3963 986
8713 836
71680 825
08881 819
6023 796
V1052 793
41012 677
53551 675
73007 663
9583 654
72669 653
